# Notebook 3: Model Evaluation & Comparison

Computes mAP, IoU, Precision-Recall curves, and error analysis for all three models.
Generates the final comparison table required for the report.


In [4]:
import sys, time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torchvision.transforms.functional as TF
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
RUNS_DIR     = PROJECT_ROOT / 'runs'
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES    = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
NUM_CLASSES = len(CLASSES) + 1
CONF_THRESH = 0.25
IOU_THRESH  = 0.50
IMG_SIZE    = 640

c:\Users\ahuru\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Keep data.yaml pointed at the local data folder
import yaml as _yaml

_yaml_path = PROJECT_ROOT / 'data' / 'data.yaml'
_cfg = _yaml.safe_load(_yaml_path.read_text())
_cfg['path'] = str(PROJECT_ROOT / 'data')
_yaml_path.write_text(_yaml.dump(_cfg, sort_keys=False))
print(f'data.yaml path set to: {_cfg["path"]}')


data.yaml path set to: D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\data


## 1. Load Models


In [20]:
import csv as _csv
from ultralytics import YOLO
from models.ssdlite_detector import build_model as build_ssd


def _best_yolo_weights(model_dir: Path):
    """Return best.pt from the vN subfolder with highest mAP@0.5 (v1, v2, ... naming)."""
    versions = sorted(
        [d for d in model_dir.iterdir() if d.is_dir() and d.name.startswith('v')],
        key=lambda d: int(d.name[1:])
    )
    if not versions:
        raise FileNotFoundError(f'No v* runs in {model_dir}')
    best_ver, best_map = versions[-1], -1.0
    for v in versions:
        results = v / 'results.csv'
        if not results.exists() or results.stat().st_size == 0:
            continue
        try:
            with open(results) as f:
                rows = list(_csv.DictReader(f))
            v_best = max(float(r.get('metrics/mAP50(B)', 0)) for r in rows) if rows else 0.0
            if v_best > best_map:
                best_map, best_ver = v_best, v
        except Exception:
            continue
    return best_ver / 'weights' / 'best.pt'


def _best_yolo_weights_named(model_dir: Path):
    """Return best.pt from a named subfolder (e.g. smartcampus_640) with highest mAP@0.5."""
    subdirs = [d for d in model_dir.iterdir() if d.is_dir() and not d.name.startswith('v')]
    if not subdirs:
        raise FileNotFoundError(f'No named runs in {model_dir}')
    best_dir, best_map = subdirs[-1], -1.0
    for d in subdirs:
        results = d / 'results.csv'
        if not results.exists() or results.stat().st_size == 0:
            continue
        try:
            with open(results) as f:
                rows = list(_csv.DictReader(f))
            v_best = max(float(r.get('metrics/mAP50(B)', 0)) for r in rows) if rows else 0.0
            if v_best > best_map:
                best_map, best_dir = v_best, d
        except Exception:
            continue
    return best_dir / 'weights' / 'best.pt'


def _latest_ssd_weights(model_dir: Path):
    ckpts = sorted(model_dir.glob('best_v*.pth'), key=lambda p: int(p.stem.split('_v')[1]))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoints in {model_dir}')
    return ckpts[-1]


models = {}
# Skip local val for checkpoints whose class order differs from data.yaml
inference_only = set()

# Load local-order YOLO checkpoints that can use model.val
for model_name, model_dir in [('YOLOv11n', RUNS_DIR / 'yolo11n'),
                               ('YOLOv8n',  RUNS_DIR / 'yolov8n')]:
    if model_dir.exists():
        try:
            w = _best_yolo_weights(model_dir)
            models[model_name] = YOLO(str(w))
            print(f'Loaded {model_name} from {w}')
        except (FileNotFoundError, RuntimeError) as e:
            print(f'{model_name} skipped: {e}')

# Load RT-DETR for prediction checks only because its class order differs
rtdetr_dir = RUNS_DIR / 'rtdetr_l'
if rtdetr_dir.exists():
    try:
        w = _best_yolo_weights(rtdetr_dir)
        models['RT-DETR-L'] = YOLO(str(w))
        inference_only.add('RT-DETR-L')
        print(f'Loaded RT-DETR-L from {w}')
        print('  Note: RT-DETR-L uses inference only because class order differs')
    except (FileNotFoundError, RuntimeError) as e:
        print(f'RT-DETR-L skipped: {e}')

# Load Kaggle-order YOLO checkpoints for prediction checks only
for model_name, model_dir in [('YOLOv11s', RUNS_DIR / 'yolo11s'),
                               ('YOLOv8s',  RUNS_DIR / 'yolov8s')]:
    if model_dir.exists():
        try:
            w = _best_yolo_weights_named(model_dir)
            models[model_name] = YOLO(str(w))
            inference_only.add(model_name)
            print(f'Loaded {model_name} from {w}')
            print(f'  Class order differs for {model_name}, using prediction checks only')
        except (FileNotFoundError, RuntimeError) as e:
            print(f'{model_name} skipped: {e}')

# SSDLite
ssd_dir = RUNS_DIR / 'ssdlite'
if ssd_dir.exists():
    try:
        w = _latest_ssd_weights(ssd_dir)
        ssd = build_ssd(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
        ckpt = torch.load(str(w), map_location=DEVICE)
        ssd.load_state_dict(ckpt['model_state_dict'])
        ssd.eval()
        models['SSDLite'] = ssd
        print(f'Loaded SSDLite from {w}')
    except FileNotFoundError as e:
        print(f'SSDLite: {e}')

print(f'\nModels loaded: {list(models.keys())}')
print(f'Prediction-only models: {sorted(inference_only)}')

Loaded YOLOv11n from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\yolo11n\v1\weights\best.pt
Loaded YOLOv8n from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\yolov8n\v1\weights\best.pt
Loaded RT-DETR-L from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\rtdetr_l\v1\weights\best.pt
  Note: RT-DETR-L uses inference only because class order differs
Loaded YOLOv11s from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\yolo11s\smartcampus_640\weights\best.pt
  Class order differs for YOLOv11s, using prediction checks only
Loaded YOLOv8s from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\yolov8s\smartcampus_640\weights\best.pt
  Class order differs for YOLOv8s, using prediction checks only

Models loaded: ['YOLOv11n', 'YOLOv8n', 'RT-DETR-L', 'YOLOv11s', 'YOLOv8s']
Prediction-only models: ['RT-DETR-L', 'YOLOv11s', 'YOLOv8s']


In [21]:
# Map Kaggle class IDs to the local label order used by data.yaml
LOCAL_ORDER = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
KAGGLE_ORDER = ['corrosion', 'cracks', 'paint_degradation', 'potholes', 'spalling']

KAGGLE_TO_LOCAL = {i: LOCAL_ORDER.index(cls) for i, cls in enumerate(KAGGLE_ORDER)}
print('Kaggle to local class ID mapping:', KAGGLE_TO_LOCAL)

MODEL_CLASS_REMAP = {name: KAGGLE_TO_LOCAL for name in inference_only}
print('Models using class remap:', sorted(MODEL_CLASS_REMAP.keys()))

# Run detailed checks on the strongest models first
ANALYSIS_ORDER = ['YOLOv11s', 'YOLOv8s', 'RT-DETR-L', 'YOLOv8n', 'YOLOv11n', 'YOLOv8n_v2']
YOLO_MODELS = [(n, models[n]) for n in ANALYSIS_ORDER if n in models]
print('Models available for detailed checks:', [n for n, _ in YOLO_MODELS])


Kaggle to local class ID mapping: {0: 2, 1: 0, 2: 4, 3: 3, 4: 1}
Models using class remap: ['RT-DETR-L', 'YOLOv11s', 'YOLOv8s']
Models available for detailed checks: ['YOLOv11s', 'YOLOv8s', 'RT-DETR-L', 'YOLOv8n', 'YOLOv11n']


In [22]:
import subprocess, sys, cv2
import torchvision.transforms.functional as TF

# Install effdet only when EfficientDet weights are available
try:
    import effdet as _effdet_check
except ModuleNotFoundError:
    print('Installing effdet')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'effdet', '-q'], check=False)
    print('effdet installed.')

# EfficientDet-D0 model setup
EFFDET_WEIGHT = RUNS_DIR / 'efficientdet_d0' / 'v1' / 'weights' / 'best.pth'
effdet_model = None
if EFFDET_WEIGHT.exists():
    try:
        from effdet import create_model, DetBenchPredict
        _net = create_model('tf_efficientdet_d0', pretrained=False,
                            num_classes=5, image_size=(IMG_SIZE, IMG_SIZE))
        _ckpt = torch.load(str(EFFDET_WEIGHT), map_location=DEVICE, weights_only=False)
        _net.load_state_dict(_ckpt['model_state_dict'])
        effdet_model = DetBenchPredict(_net).to(DEVICE).eval()
        print(f'Loaded EfficientDet-D0 (epoch {_ckpt["epoch"]}) from {EFFDET_WEIGHT}')
    except Exception as e:
        print(f'EfficientDet-D0 skipped: {e}')
else:
    print(f'EfficientDet-D0 weights not found at {EFFDET_WEIGHT}')

# Faster R-CNN model setup
FRCNN_WEIGHT = RUNS_DIR / 'faster_rcnn' / 'v1' / 'weights' / 'best.pth'
frcnn_model = None
if FRCNN_WEIGHT.exists():
    try:
        from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
        from torchvision.models.detection import FasterRCNN
        from torchvision.models import ResNet18_Weights
        _backbone = resnet_fpn_backbone('resnet18', weights=ResNet18_Weights.DEFAULT,
                                        trainable_layers=3)
        frcnn_model = FasterRCNN(_backbone, num_classes=6).to(DEVICE)
        _sd = torch.load(str(FRCNN_WEIGHT), map_location=DEVICE, weights_only=True)
        frcnn_model.load_state_dict(_sd)
        frcnn_model.eval()
        print(f'Loaded Faster R-CNN (ResNet18-FPN) from {FRCNN_WEIGHT}')
    except Exception as e:
        print(f'Faster R-CNN skipped: {e}')
else:
    print(f'Faster R-CNN weights not found at {FRCNN_WEIGHT}')

# Shared prediction wrappers keep outputs in the local class order

_EFFDET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_EFFDET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# EfficientDet class IDs are 1-indexed in local order
# Faster R-CNN class IDs are mapped from its training label order
_FRCNN_TO_LOCAL = {1: 2, 2: 0, 3: 4, 4: 3, 5: 1}


def _yolo_predict_fn(model, class_remap=None):
    def predict(img_path, conf):
        result = model.predict(str(img_path), conf=conf, verbose=False)[0]
        orig_hw = result.orig_shape
        boxes   = [b.xyxy[0].tolist() for b in result.boxes]
        raw_ids = [int(b.cls.item()) for b in result.boxes]
        cls_ids = [class_remap.get(c, c) if class_remap else c for c in raw_ids]
        return boxes, cls_ids, orig_hw
    return predict


def _effdet_predict_fn(model):
    def predict(img_path, conf):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            return [], [], (IMG_SIZE, IMG_SIZE)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        orig_hw = img_rgb.shape[:2]
        img_r   = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_n   = (img_r.astype(np.float32) / 255.0 - _EFFDET_MEAN) / _EFFDET_STD
        t = torch.tensor(img_n).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            outputs = model(t)
        det = outputs[0].cpu().numpy()
        det = det[det[:, 4] > conf]
        if len(det) == 0:
            return [], [], orig_hw
        sh, sw = orig_hw[0] / IMG_SIZE, orig_hw[1] / IMG_SIZE
        boxes, cls_ids = [], []
        for row in det:
            ymin, xmin, ymax, xmax = row[:4]
            boxes.append([xmin * sw, ymin * sh, xmax * sw, ymax * sh])
            raw_id = int(row[5]) - 1  # effdet is 1-indexed in local order
            cls_ids.append(max(0, min(raw_id, len(CLASSES) - 1)))
        return boxes, cls_ids, orig_hw
    return predict


def _frcnn_predict_fn(model):
    def predict(img_path, conf):
        img = Image.open(str(img_path)).convert('RGB')
        orig_hw = (img.height, img.width)
        t = TF.to_tensor(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            preds = model(t)[0]
        mask = preds['scores'] >= conf
        raw_boxes = [b.tolist() for b in preds['boxes'][mask]]
        raw_ids   = [int(l) for l in preds['labels'][mask]]
        boxes, cls_ids = [], []
        for b, r in zip(raw_boxes, raw_ids):
            c = _FRCNN_TO_LOCAL.get(r, -1)
            if 0 <= c < len(CLASSES):
                boxes.append(b)
                cls_ids.append(c)
        return boxes, cls_ids, orig_hw
    return predict


# Build the shared model list for evaluation
ALL_MODELS = []
for _name, _model in YOLO_MODELS:
    ALL_MODELS.append((_name, _yolo_predict_fn(_model, MODEL_CLASS_REMAP.get(_name))))

if effdet_model is not None:
    ALL_MODELS.append(('EfficientDet-D0', _effdet_predict_fn(effdet_model)))

if frcnn_model is not None:
    ALL_MODELS.append(('Faster R-CNN', _frcnn_predict_fn(frcnn_model)))

print(f'\nModels for detailed checks: {[n for n, _ in ALL_MODELS]}')

Loaded EfficientDet-D0 (epoch 26) from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\efficientdet_d0\v1\weights\best.pth


c:\Users\ahuru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Loaded Faster R-CNN (ResNet18-FPN) from D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\runs\faster_rcnn\v1\weights\best.pth

Models for detailed checks: ['YOLOv11s', 'YOLOv8s', 'RT-DETR-L', 'YOLOv8n', 'YOLOv11n', 'EfficientDet-D0', 'Faster R-CNN']


## 2. Evaluation Functions


In [10]:
from torch.utils.data import DataLoader, Dataset

IMAGE_EXTS = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']


class TestDataset(Dataset):
    def __init__(self, split: str = 'test'):
        self.img_dir = DATA_DIR / 'images' / split
        self.lbl_dir = DATA_DIR / 'labels' / split
        seen, imgs = set(), []
        for ext in IMAGE_EXTS:
            for p in self.img_dir.glob(ext):
                if p.name.lower() not in seen:
                    seen.add(p.name.lower())
                    imgs.append(p)
        self.images = sorted(imgs)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        tensor = TF.to_tensor(img)
        boxes, labels = [], []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * IMG_SIZE)
                y1 = max(0.0, (cy - bh / 2) * IMG_SIZE)
                x2 = min(float(IMG_SIZE), (cx + bw / 2) * IMG_SIZE)
                y2 = min(float(IMG_SIZE), (cy + bh / 2) * IMG_SIZE)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)
        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros(0,      dtype=torch.int64)
        return tensor, {'boxes': boxes_t, 'labels': labels_t, 'image_id': torch.tensor([idx])}


def collate_fn(batch):
    return tuple(zip(*batch))


test_ds     = TestDataset('test')
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Test images: {len(test_ds)}')

Test images: 965


In [12]:
@torch.no_grad()
def evaluate_torchvision(model, loader, device, conf=CONF_THRESH):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    times = []
    for images, targets in tqdm(loader):
        imgs = [img.to(device) for img in images]
        t0 = time.time()
        outputs = model(imgs)
        times.append((time.time() - t0) / len(imgs) * 1000)  # ms/image
        preds = []
        for o in outputs:
            mask = o['scores'] >= conf
            preds.append({'boxes':  o['boxes'][mask].cpu(),
                          'scores': o['scores'][mask].cpu(),
                          'labels': o['labels'][mask].cpu()})
        gts = [{'boxes':  t['boxes'].cpu(),
                'labels': t['labels'].cpu()} for t in targets]
        metric.update(preds, gts)
    result = metric.compute()
    result['inference_ms'] = np.mean(times)
    return result


def evaluate_yolo(model, data_yaml, split='test'):
    return model.val(data=data_yaml, split=split)


def measure_yolo_inference_speed(model, n_images: int = 50):
    imgs = list((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:n_images]
    times = []
    for p in imgs:
        t0 = time.time()
        model.predict(str(p), verbose=False)
        times.append((time.time() - t0) * 1000)
    return np.mean(times)

## 3. Run Evaluation on Test Set


In [13]:
DATA_YAML = str(DATA_DIR / 'data.yaml')
results = {}

for model_name, model in models.items():
    if model_name in inference_only:
        print(f'\nSkipping {model_name} because class order differs from local labels')
        continue
    if model_name.startswith('YOLO') or model_name == 'RT-DETR-L':
        print(f'\nEvaluating {model_name}')
        val   = model.val(data=DATA_YAML, split='test')
        speed = measure_yolo_inference_speed(model)
        results[model_name] = {
            'mAP@0.5':      round(float(val.box.map50), 4),
            'mAP@0.5:0.95': round(float(val.box.map),   4),
            'Precision':    round(float(val.box.mp),     4),
            'Recall':       round(float(val.box.mr),     4),
            'Speed (ms)':   round(speed, 2),
        }
    elif model_name == 'SSDLite':
        print(f'\nEvaluating {model_name}')
        r = evaluate_torchvision(model, test_loader, DEVICE)
        results[model_name] = {
            'mAP@0.5':      round(float(r['map_50']), 4),
            'mAP@0.5:0.95': round(float(r['map']),    4),
            'Precision':    None,
            'Recall':       None,
            'Speed (ms)':   round(float(r['inference_ms']), 2),
        }

print('\nLocal val results (models with matching class order only):')
for m, r in results.items():
    print(f'  {m}: mAP@0.5={r["mAP@0.5"]}, mAP@0.5:0.95={r["mAP@0.5:0.95"]}')


Evaluating YOLOv11n
Ultralytics 8.4.61  Python-3.12.7 torch-2.9.1+cpu CPU (AMD Ryzen 5 3500U with Radeon Vega Mobile Gfx)
val: Fast image access  (ping: 0.10.0 ms, read: 54.324.2 MB/s, size: 58.4 KB)
val: Scanning D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\data\labels\test.cache... 965 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 965/965  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 61/61 3.3s/it 3:204.3ss
                   all        965       1497      0.161      0.166      0.127      0.059
                cracks        172        312      0.364      0.484      0.347      0.167
              spalling        194        292      0.056     0.0342     0.0118     0.0028
             corrosion        300        300          0          0          0          0
              potholes        154        277      0.387      0.312      0.279      0.126
     paint_degradation        217        316      

## 4. Model Comparison Table


In [9]:
# Load reported metrics so Kaggle-order checkpoints are not scored with local labels
csv_path = RUNS_DIR / 'model_comparison.csv'
comparison_df = pd.read_csv(csv_path, index_col='Model')

print('\nModel comparison table')
print(comparison_df.to_string())

# Sort models by mAP before plotting
plot_df = comparison_df.copy()
plot_df['mAP@0.5'] = pd.to_numeric(plot_df['mAP@0.5'], errors='coerce')
plot_df['mAP@0.5:0.95'] = pd.to_numeric(plot_df['mAP@0.5:0.95'], errors='coerce')
plot_df = plot_df.dropna(subset=['mAP@0.5']).sort_values('mAP@0.5', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics_to_plot = ['mAP@0.5', 'mAP@0.5:0.95']
bar_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, metric in enumerate(metrics_to_plot):
    vals  = plot_df[metric].dropna()
    names = vals.index.tolist()
    axes[i].bar(names, vals.values, color=bar_colors[:len(names)], edgecolor='black')
    axes[i].set_title(metric, fontsize=12)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Score')
    axes[i].tick_params(axis='x', rotation=30)
    for j, v in enumerate(vals.values):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Model Comparison - All Models (Training-Time Metrics)', fontsize=14)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png and kept model_comparison.csv unchanged')


Model comparison table
                 mAP@0.5  mAP@0.5:0.95  Precision  Recall      F1                                                              Notes
Model                                                                                                                               
YOLOv11s          0.7799        0.5825     0.8250  0.7029  0.7591                              Norman - Dataset 10 (3026 train imgs)
YOLOv8s           0.7587        0.5472     0.7877  0.6863  0.7335                              Norman - Dataset 10 (3026 train imgs)
EfficientDet-D0   0.7311        0.4878     0.7986  0.7581  0.7767                                     Tyra - Dataset 10 (110 epochs)
Faster R-CNN      0.6936        0.3884     0.4476  0.7595  0.5602                                                Amanda - Dataset 10
RT-DETR-L         0.5800        0.3510     0.7420     NaN  0.5800                                                 Emily - Dataset 10
YOLOv8n_v2        0.4368        0.2530     0.

<Figure size 1400x500 with 2 Axes>

Saved model_comparison.png and kept model_comparison.csv unchanged


## 5. Per-Class AP (YOLO)


In [10]:
colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

# Only run per-class AP for models whose class order matches local labels (not inference_only)
eligible = [m for m in models if (m.startswith('YOLO') or m == 'RT-DETR-L') and m not in inference_only]

for model_name in eligible:
    val_res = models[model_name].val(data=DATA_YAML, split='test')
    per_class = dict(zip(CLASSES, val_res.box.maps))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(per_class.keys()), list(per_class.values()), color=colors_bar, edgecolor='black')
    ax.set_title(f'{model_name} - Per-Class AP@0.5')
    ax.set_ylabel('AP@0.5')
    ax.set_ylim(0, 1)
    for i, (k, v) in enumerate(per_class.items()):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    fname = f'per_class_ap_{model_name.lower().replace("-", "_")}.png'
    plt.savefig(str(RUNS_DIR / fname), dpi=150)
    plt.show()
    print(f'Saved {fname}')

if not eligible:
    print('No models with local class order available for per-class AP - '
          'using Norman\'s pre-computed per-class CSV instead.')
    for csv_name in ['YOLOv11s_per_class.csv', 'YOLOv8s_per_class.csv']:
        p = RUNS_DIR / csv_name
        if p.exists():
            df = pd.read_csv(p)
            print(f'\n{csv_name}:')
            print(df.to_string(index=False))

Ultralytics 8.4.61  Python-3.12.7 torch-2.9.1+cpu CPU (AMD Ryzen 5 3500U with Radeon Vega Mobile Gfx)
val: Fast image access  (ping: 0.10.0 ms, read: 38.924.7 MB/s, size: 51.4 KB)
                   all        965       1497      0.161      0.166      0.127      0.059
                cracks        172        312      0.364      0.484      0.347      0.167
              spalling        194        292      0.056     0.0342     0.0118     0.0028
             corrosion        300        300          0          0          0          0
              potholes        154        277      0.387      0.312      0.279      0.126
     paint_degradation        217        316          0          0   0.000181   4.21e-05
Speed: 2.4ms preprocess, 423.4ms inference, 0.0ms loss, 4.6ms postprocess per image
Results saved to D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\notebooks\runs\detect\val-9


<Figure size 800x400 with 1 Axes>

Saved per_class_ap_yolov11n.png
Ultralytics 8.4.61  Python-3.12.7 torch-2.9.1+cpu CPU (AMD Ryzen 5 3500U with Radeon Vega Mobile Gfx)
val: Fast image access  (ping: 0.10.0 ms, read: 34.921.9 MB/s, size: 56.6 KB)
                   all        965       1497      0.187      0.164      0.121     0.0549
                cracks        172        312      0.328      0.478      0.271      0.134
              spalling        194        292     0.0409     0.0308    0.00935     0.0022
             corrosion        300        300          0          0          0          0
              potholes        154        277      0.553       0.31      0.324      0.137
     paint_degradation        217        316      0.012    0.00316   0.000821    0.00026
Speed: 1.9ms preprocess, 231.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to D:\.Adil\uni\SEM 5\AI in ENG\Final_Project\ai_for_engineering\notebooks\runs\detect\val-10


<Figure size 800x400 with 1 Axes>

Saved per_class_ap_yolov8n.png


## 6. Severity Distribution on Test Set


In [11]:
def severity_label(bbox_w_norm: float, bbox_h_norm: float):
    """Returns Low/Medium/High based on fractional bbox area."""
    area_pct = bbox_w_norm * bbox_h_norm * 100
    if area_pct < 5:
        return 'Low'
    elif area_pct <= 20:
        return 'Medium'
    return 'High'


severity_counts = {'Low': 0, 'Medium': 0, 'High': 0}
for lbl_path in (DATA_DIR / 'labels' / 'test').glob('*.txt'):
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            bw, bh = float(parts[3]), float(parts[4])
            severity_counts[severity_label(bw, bh)] += 1

print('Ground-truth severity distribution in test set:')
for sev, cnt in severity_counts.items():
    print(f'  {sev}: {cnt}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(severity_counts.values(), labels=severity_counts.keys(),
       autopct='%1.1f%%', colors=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Ground-Truth Severity Distribution (Test Set)')
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'severity_distribution.png'), dpi=150)
plt.show()

Ground-truth severity distribution in test set:
  Low: 763
  Medium: 354
  High: 380


<Figure size 500x400 with 1 Axes>

## 7. Qualitative Prediction Visualisation (YOLO)


In [12]:
if 'YOLOv11n' in models:
    CLASS_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
    test_imgs = sorted((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, img_path in enumerate(test_imgs):
        result = models['YOLOv11n'].predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        w, h = img.size
        for box in result.boxes:
            cls   = int(box.cls.item())
            conf  = float(box.conf.item())
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            bw_n = (x2 - x1) / w
            bh_n = (y2 - y1) / h
            sev  = severity_label(bw_n, bh_n)
            label = f'{CLASSES[cls]} {conf:.2f} [{sev}]'
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2,
                                       edgecolor=CLASS_COLORS[cls],
                                       facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-5, label, color=CLASS_COLORS[cls],
                         fontsize=7, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(img_path.name, fontsize=8)

    plt.suptitle('YOLOv11n Predictions with Severity', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(RUNS_DIR / 'prediction_samples.png'), dpi=120)
    plt.show()

<Figure size 1500x800 with 6 Axes>

## 8. IoU Distribution Histogram

For every predicted box that matches a ground-truth box (IoU >= 0.5), we record the exact IoU value.
This shows *how tightly* the model localises defects - not just whether it detects them.
A model that mostly scores 0.5-0.6 is "barely passing", one clustered near 1.0 localises precisely.


In [23]:
def compute_iou(box_a, box_b):
    ix1 = max(box_a[0], box_b[0])
    iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2])
    iy2 = min(box_a[3], box_b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def collect_iou_scores(predict_fn, img_dir, lbl_dir, conf=CONF_THRESH, max_images=300):
    """
    Match each prediction to the best GT box (IoU >= IOU_THRESH) and record the IoU.
    predict_fn returns boxes, local class IDs and original image size
    Class remapping is handled inside predict_fn.
    Returns {class_name: [iou_values]}.
    """
    iou_by_class = {c: [] for c in CLASSES}
    img_paths = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.jpeg')) +
                       list(img_dir.glob('*.png')))[:max_images]

    for img_path in tqdm(img_paths, desc='IoU', leave=False):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue

        pred_boxes, cls_ids, orig_hw = predict_fn(img_path, conf)
        h, w = orig_hw

        gt_boxes, gt_classes = [], []
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
            gt_boxes.append([(cx - bw/2)*w, (cy - bh/2)*h,
                              (cx + bw/2)*w, (cy + bh/2)*h])
            gt_classes.append(cls)

        if not gt_boxes:
            continue

        matched_gt = set()
        for pred_box, cls_id in zip(pred_boxes, cls_ids):
            best_iou, best_gi = 0.0, -1
            for gi, gb in enumerate(gt_boxes):
                if gi in matched_gt:
                    continue
                iou = compute_iou(pred_box, gb)
                if iou > best_iou:
                    best_iou, best_gi = iou, gi
            if best_iou >= IOU_THRESH and best_gi >= 0:
                matched_gt.add(best_gi)
                if 0 <= cls_id < len(CLASSES):
                    iou_by_class[CLASSES[cls_id]].append(best_iou)

    return iou_by_class


# Collect matched IoU values for each detector
all_model_iou = {}
for model_name, predict_fn in ALL_MODELS:
    print(f'Collecting IoU for {model_name}')
    scores = collect_iou_scores(predict_fn,
                                DATA_DIR / 'images' / 'test',
                                DATA_DIR / 'labels' / 'test')
    all_model_iou[model_name] = scores
    total = sum(len(v) for v in scores.values())
    per_cls = '  '.join(f'{c[:4]}: n={len(v)} mean={np.mean(v):.3f}' for c, v in scores.items() if v)
    print(f'  matched={total}  {per_cls}')

print('\nIoU scores collected.')

  matched=152  crac: n=14 mean=0.693  spal: n=138 mean=0.886


  matched=158  crac: n=20 mean=0.705  spal: n=134 mean=0.863  poth: n=1 mean=0.509  pain: n=3 mean=0.884


  matched=160  crac: n=18 mean=0.610  spal: n=139 mean=0.858  corr: n=1 mean=0.561  poth: n=1 mean=0.633  pain: n=1 mean=0.852


  matched=82  crac: n=71 mean=0.754  spal: n=9 mean=0.614  poth: n=2 mean=0.630


  matched=72  crac: n=58 mean=0.765  spal: n=9 mean=0.614  corr: n=2 mean=0.792  poth: n=3 mean=0.596


  matched=22  spal: n=4 mean=0.621  pain: n=18 mean=0.699


  matched=177  crac: n=29 mean=0.654  spal: n=144 mean=0.804  poth: n=2 mean=0.630  pain: n=2 mean=0.766

IoU scores collected.


In [24]:
colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

# Plot one overall IoU distribution for each model
n = len(all_model_iou)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
if n == 1:
    axes = [axes]

for ax, (mname, scores) in zip(axes, all_model_iou.items()):
    all_ious = [v for cls_v in scores.values() for v in cls_v]
    if all_ious:
        ax.hist(all_ious, bins=20, range=(0.5, 1.0), color='steelblue', edgecolor='black', alpha=0.85)
        ax.axvline(np.mean(all_ious), color='red', linestyle='--',
                   label=f'mean={np.mean(all_ious):.3f}')
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No matched\npredictions', ha='center', va='center',
                transform=ax.transAxes, fontsize=9)
    ax.set_title(f'{mname}\n(n={len(all_ious)})', fontsize=9)
    ax.set_xlabel('IoU')
    ax.set_xlim(0.5, 1.0)

axes[0].set_ylabel('Count')
plt.suptitle('IoU Distribution of Matched Predictions - All Models (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'iou_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved iou_histogram.png')

# Plot per-class IoU breakdowns for each model
for mname, scores in all_model_iou.items():
    all_ious = [v for cls_v in scores.values() for v in cls_v]
    fig2, axes2 = plt.subplots(2, 3, figsize=(15, 7))
    axes2 = axes2.flatten()

    axes2[0].hist(all_ious, bins=20, range=(0.5, 1.0), color='steelblue', edgecolor='black')
    axes2[0].set_title(f'All Classes (n={len(all_ious)})', fontsize=10)
    axes2[0].set_xlabel('IoU')
    axes2[0].set_ylabel('Count')
    if all_ious:
        axes2[0].axvline(np.mean(all_ious), color='red', linestyle='--',
                         label=f'mean={np.mean(all_ious):.3f}')
        axes2[0].legend()
    axes2[0].set_xlim(0.5, 1.0)

    for i, (cls, cls_scores) in enumerate(scores.items(), start=1):
        ax = axes2[i]
        if cls_scores:
            ax.hist(cls_scores, bins=15, range=(0.5, 1.0),
                    color=colors_bar[i - 1], edgecolor='black', alpha=0.85)
            ax.axvline(np.mean(cls_scores), color='red', linestyle='--',
                       label=f'mean={np.mean(cls_scores):.3f}')
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No matched\npredictions', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9)
        ax.set_title(f'{cls} (n={len(cls_scores)})', fontsize=9)
        ax.set_xlabel('IoU')
        ax.set_ylabel('Count')
        ax.set_xlim(0.5, 1.0)

    plt.suptitle(f'{mname} - Per-Class IoU Distribution (Test Set)', fontsize=12)
    plt.tight_layout()
    fname = f'iou_histogram_{mname.lower().replace("-", "_").replace(".", "_").replace(" ", "_")}.png'
    plt.savefig(str(RUNS_DIR / fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

<Figure size 2800x400 with 7 Axes>

Saved iou_histogram.png


<Figure size 1500x700 with 6 Axes>

Saved iou_histogram_yolov11s.png


<Figure size 1500x700 with 6 Axes>

...

Saved iou_histogram_yolov11n.png


<Figure size 1500x700 with 6 Axes>

Saved iou_histogram_efficientdet_d0.png


<Figure size 1500x700 with 6 Axes>

Saved iou_histogram_faster_r_cnn.png


## 9. False Positive / False Negative Error Analysis

- **False Positive (FP):** model predicts a defect where there is none (or wrong class / poor localisation)
- **False Negative (FN):** a real defect exists but the model missed it entirely

Understanding *why* errors occur is required for the evaluation rubric (detailed analysis, model limitations).
We find the worst-case examples automatically and visualise them.


In [25]:
def collect_fp_fn(predict_fn, img_dir, lbl_dir, conf=CONF_THRESH, iou_thresh=IOU_THRESH,
                  max_images=300):
    """
    For each test image return FP records (predictions with no matching GT) and
    FN records (GT boxes with no matching prediction).
    predict_fn returns boxes, local class IDs and original image size
    Class remapping is handled inside predict_fn.
    """
    fp_records, fn_records = [], []
    fp_by_class = {c: 0 for c in CLASSES}
    fn_by_class = {c: 0 for c in CLASSES}

    img_paths = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.jpeg')) +
                       list(img_dir.glob('*.png')))[:max_images]

    for img_path in tqdm(img_paths, desc='FP/FN', leave=False):
        lbl_path = lbl_dir / (img_path.stem + '.txt')

        pred_boxes, pred_classes, orig_hw = predict_fn(img_path, conf)
        h, w = orig_hw

        gt_boxes, gt_classes = [], []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                gt_boxes.append([(cx - bw/2)*w, (cy - bh/2)*h,
                                  (cx + bw/2)*w, (cy + bh/2)*h])
                gt_classes.append(cls)

        matched_gt, matched_pred = set(), set()
        for pi, (pb, pc) in enumerate(zip(pred_boxes, pred_classes)):
            best_iou, best_gi = 0.0, -1
            for gi, gb in enumerate(gt_boxes):
                if gi in matched_gt:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou, best_gi = iou, gi
            if best_iou >= iou_thresh:
                matched_gt.add(best_gi)
                matched_pred.add(pi)

        fp_boxes, fp_cls = [], []
        for pi, (pb, pc) in enumerate(zip(pred_boxes, pred_classes)):
            if pi not in matched_pred:
                fp_boxes.append(pb)
                fp_cls.append(pc)
                if 0 <= pc < len(CLASSES):
                    fp_by_class[CLASSES[pc]] += 1

        fn_boxes, fn_cls = [], []
        for gi, (gb, gc) in enumerate(zip(gt_boxes, gt_classes)):
            if gi not in matched_gt:
                fn_boxes.append(gb)
                fn_cls.append(gc)
                if 0 <= gc < len(CLASSES):
                    fn_by_class[CLASSES[gc]] += 1

        if fp_boxes:
            fp_records.append((img_path, fp_boxes, fp_cls, len(fp_boxes)))
        if fn_boxes:
            fn_records.append((img_path, fn_boxes, fn_cls, len(fn_boxes)))

    fp_records.sort(key=lambda x: x[3], reverse=True)
    fn_records.sort(key=lambda x: x[3], reverse=True)
    return fp_records, fn_records, fp_by_class, fn_by_class


# Count false positives and false negatives for each detector
all_model_fp_fn = {}
for model_name, predict_fn in ALL_MODELS:
    print(f'Checking FP and FN for {model_name}')
    fp_rec, fn_rec, fp_cls, fn_cls = collect_fp_fn(predict_fn,
                                                    DATA_DIR / 'images' / 'test',
                                                    DATA_DIR / 'labels' / 'test')
    all_model_fp_fn[model_name] = (fp_rec, fn_rec, fp_cls, fn_cls)
    total_fp = sum(fp_cls.values())
    total_fn = sum(fn_cls.values())
    print(f'  Total FP={total_fp}  FN={total_fn}')

print('\nFP and FN counts collected.')

Checking FP and FN for YOLOv11s


  Total FP=244  FN=321
Checking FP and FN for YOLOv8s


  Total FP=262  FN=315
Checking FP and FN for RT-DETR-L


  Total FP=1147  FN=313
Checking FP and FN for YOLOv8n


  Total FP=285  FN=391
Checking FP and FN for YOLOv11n


  Total FP=184  FN=401
Checking FP and FN for EfficientDet-D0


  Total FP=501  FN=451
Checking FP and FN for Faster R-CNN


  Total FP=885  FN=296

FP and FN counts collected.


In [26]:
import numpy as _np

model_names = list(all_model_fp_fn.keys())
x = np.arange(len(CLASSES))
width = 0.8 / len(model_names)
palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, err_type in zip(axes, ['FP', 'FN']):
    idx = 2 if err_type == 'FP' else 3  # Pick FP or FN class-count dictionary
    for i, mname in enumerate(model_names):
        counts = all_model_fp_fn[mname][idx]
        vals = [counts[c] for c in CLASSES]
        offset = (i - len(model_names) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=mname,
                      color=palette[i % len(palette)], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{err_type} Counts per Class - All Models', fontsize=11)
    ax.set_ylabel('Count')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, rotation=20, ha='right')
    ax.legend(fontsize=7)

plt.suptitle('False Positive / False Negative Comparison (Test Set, 300 images)', fontsize=13)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'fp_fn_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fp_fn_by_class.png')

# Summary table
print('\nFP and FN summary')
header = f"{'Model':<14}" + ''.join(f'{c[:6]:>8}' for c in CLASSES) + f"{'TOTAL':>8}"
print(header)
for err_type, idx in [('FP', 2), ('FN', 3)]:
    print(f'\n{err_type}')
    for mname in model_names:
        row = all_model_fp_fn[mname][idx]
        vals = [row[c] for c in CLASSES]
        print(f"  {mname:<14}" + ''.join(f'{v:>8}' for v in vals) + f"{sum(vals):>8}")

<Figure size 1600x500 with 2 Axes>

Saved fp_fn_by_class.png

FP and FN summary
Model           cracks  spalli  corros  pothol  paint_   TOTAL

FP
  YOLOv11s            62     165      11       3       3     244
  YOLOv8s             72     170       1      12       7     262
  RT-DETR-L          624     362      30      49      82    1147
  YOLOv8n             95      64      66      13      47     285
  YOLOv11n            58      48      32      10      36     184
  EfficientDet-D0      31     102      43      25     300     501
  Faster R-CNN       228     461      34      93      69     885

FN
  YOLOv11s           158     157       6       0       0     321
  YOLOv8s            150     159       6       0       0     315
  RT-DETR-L          151     156       6       0       0     313
  YOLOv8n            106     279       6       0       0     391
  YOLOv11n           115     280       6       0       0     401
  EfficientDet-D0     171     274       6       0       0     451
  Faster R-CNN       138     152      

In [27]:
def plot_error_examples(records, error_type, model_name, n=6):
    top = records[:n]
    if not top:
        print(f'No {error_type} examples for {model_name}.')
        return
    cols = min(3, len(top))
    rows = (len(top) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten() if len(top) > 1 else [axes]
    color = '#e74c3c' if error_type == 'FP' else '#3498db'

    for i, (img_path, err_boxes, err_cls, count) in enumerate(top):
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        for box, cls_id in zip(err_boxes, err_cls):
            x1, y1, x2, y2 = box
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=2, edgecolor=color, facecolor='none',
                                      linestyle='--')
            axes[i].add_patch(rect)
            label = CLASSES[cls_id] if 0 <= cls_id < len(CLASSES) else 'unknown'
            axes[i].text(x1, y1 - 4, f'{error_type}: {label}',
                         color=color, fontsize=7, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(f'{img_path.name}\n({count} {error_type}s)', fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'{model_name}: Worst {error_type} Examples', fontsize=12)
    plt.tight_layout()
    safe = model_name.lower().replace('-', '_').replace('.', '_').replace(' ', '_')
    fname = f'{error_type.lower()}_examples_{safe}.png'
    plt.savefig(str(RUNS_DIR / fname), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')


# Save worst-case FP and FN examples for each model
for model_name, (fp_rec, fn_rec, _, _) in all_model_fp_fn.items():
    print(f'\n{model_name}')
    plot_error_examples(fp_rec, 'FP', model_name)
    plot_error_examples(fn_rec, 'FN', model_name)


YOLOv11s


<Figure size 1500x800 with 6 Axes>

Saved fp_examples_yolov11s.png


<Figure size 1500x800 with 6 Axes>

Saved fn_examples_yolov11s.png

YOLOv8s


...

Saved fn_examples_efficientdet_d0.png

Faster R-CNN


<Figure size 1500x800 with 6 Axes>

Saved fp_examples_faster_r_cnn.png


<Figure size 1500x800 with 6 Axes>

Saved fn_examples_faster_r_cnn.png


## 10. Unseen Data Generalisation Test

This section tests the model on **completely fresh images** that were never part of the training,
validation, or test splits. They were sourced independently (e.g. self-collected campus photos or new
images downloaded from the web).

**To run this section:**
1. Place your unseen images in `data/unseen/images/`
2. If you have ground-truth labels (YOLO format), place them in `data/unseen/labels/`
   (optional - qualitative results still work without labels)
3. Re-run this cell


In [32]:
def severity_label(bbox_w_norm: float, bbox_h_norm: float):
    """Returns Low/Medium/High based on fractional bbox area."""
    area_pct = bbox_w_norm * bbox_h_norm * 100
    if area_pct < 5:
        return 'Low'
    elif area_pct <= 20:
        return 'Medium'
    return 'High'

UNSEEN_IMG_DIR = DATA_DIR / 'unseen' / 'images'
UNSEEN_LBL_DIR = DATA_DIR / 'unseen' / 'labels'
UNSEEN_IMG_DIR.mkdir(parents=True, exist_ok=True)
UNSEEN_LBL_DIR.mkdir(parents=True, exist_ok=True)

unseen_imgs = sorted(list(UNSEEN_IMG_DIR.glob('*.jpg')) +
                     list(UNSEEN_IMG_DIR.glob('*.jpeg')) +
                     list(UNSEEN_IMG_DIR.glob('*.png')))

if not unseen_imgs:
    print(f'No unseen images found. Add images to: {UNSEEN_IMG_DIR}')
else:
    print(f'Found {len(unseen_imgs)} unseen images, running all models')
    CLASS_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    for model_name, predict_fn in ALL_MODELS:
        cols = min(3, len(unseen_imgs))
        rows = (len(unseen_imgs) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
        axes = np.array(axes).flatten() if len(unseen_imgs) > 1 else [axes]

        det_summary = []
        for i, img_path in enumerate(unseen_imgs):
            boxes, cls_ids, orig_hw = predict_fn(img_path, CONF_THRESH)
            img = Image.open(img_path).convert('RGB')
            axes[i].imshow(img)
            detected = set()
            for (x1, y1, x2, y2), cls_id in zip(boxes, cls_ids):
                cls_name = CLASSES[cls_id] if 0 <= cls_id < len(CLASSES) else 'unknown'
                bw_n = (x2 - x1) / img.width
                bh_n = (y2 - y1) / img.height
                sev  = severity_label(bw_n, bh_n)
                detected.add(cls_name)
                color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
                rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           linewidth=2, edgecolor=color, facecolor='none')
                axes[i].add_patch(rect)
                axes[i].text(x1, y1 - 4, f'{cls_name} [{sev}]',
                             color=color, fontsize=7, fontweight='bold')
            axes[i].axis('off')
            axes[i].set_title(f"{img_path.name} ({len(boxes)} det)", fontsize=7)
            det_summary.append((img_path.name, len(boxes), detected))

        for j in range(i + 1, len(axes)):
            axes[j].axis('off')

        plt.suptitle(f'{model_name}: Unseen Data Generalisation', fontsize=12)
        plt.tight_layout()
        safe = model_name.lower().replace('-', '_').replace('.', '_').replace(' ', '_')
        fname = f'unseen_predictions_{safe}.png'
        plt.savefig(str(RUNS_DIR / fname), dpi=120, bbox_inches='tight')
        plt.show()

        print(f'{model_name}:')
        for name, n_boxes, classes in det_summary:
            print(f'  {name}: {n_boxes} boxes - {classes if classes else "no detections"}')
        print(f'Saved {fname}')

    print('Unseen inference complete.')


Found 15 unseen images, running all models


<Figure size 1500x2000 with 15 Axes>

<Figure size 1500x2000 with 15 Axes>

YOLOv11s:
  close-up-of-a-weathered-wall-with-peeling-paint-revealing-layers-of-faded-colors-and-textured-surfaces-photo.jpg: 4 boxes - {'cracks', 'paint_degradation'}
  Concrete-Spalling.jpg: 1 boxes - {'spalling'}
  concrete-temperature-crack.jpg: 3 boxes - {'cracks'}
  images.jpg: 2 boxes - {'cracks'}
  images1.jpg: 0 boxes - no detections
  images12.jpg: 1 boxes - {'potholes'}
  images123.jpg: 2 boxes - {'potholes', 'paint_degradation'}
  images21.jpg: 1 boxes - {'corrosion'}
  images234.jpg: 6 boxes - {'corrosion'}
  images5.jpg: 1 boxes - {'cracks'}
  images99.jpg: 2 boxes - {'cracks'}
  MicrosoftTeams-image_32.jpg: 1 boxes - {'potholes'}
  Newport_Whitepit_Lane_pot_hole.JPG: 2 boxes - {'potholes'}
  shutterstock_1667846680-scaled.jpg: 3 boxes - {'corrosion'}
  What-is-Spalling-concrete.jpeg.png: 1 boxes - {'spalling'}
Saved unseen_predictions_yolov11s.png


<Figure size 1500x2000 with 15 Axes>

...

YOLOv11n:
  close-up-of-a-weathered-wall-with-peeling-paint-revealing-layers-of-faded-colors-and-textured-surfaces-photo.jpg: 1 boxes - {'paint_degradation'}
  Concrete-Spalling.jpg: 0 boxes - no detections
  concrete-temperature-crack.jpg: 0 boxes - no detections
  images.jpg: 1 boxes - {'cracks'}
  images1.jpg: 1 boxes - {'corrosion'}
  images12.jpg: 1 boxes - {'potholes'}
  images123.jpg: 1 boxes - {'spalling'}
  images21.jpg: 1 boxes - {'spalling'}
  images234.jpg: 1 boxes - {'spalling'}
  images5.jpg: 0 boxes - no detections
  images99.jpg: 1 boxes - {'cracks'}
  MicrosoftTeams-image_32.jpg: 0 boxes - no detections
  Newport_Whitepit_Lane_pot_hole.JPG: 4 boxes - {'potholes', 'cracks'}
  shutterstock_1667846680-scaled.jpg: 1 boxes - {'corrosion'}
  What-is-Spalling-concrete.jpeg.png: 2 boxes - {'spalling', 'corrosion'}
Saved unseen_predictions_yolov11n.png


<Figure size 1500x2000 with 15 Axes>

EfficientDet-D0:
  close-up-of-a-weathered-wall-with-peeling-paint-revealing-layers-of-faded-colors-and-textured-surfaces-photo.jpg: 3 boxes - {'paint_degradation', 'spalling'}
  Concrete-Spalling.jpg: 4 boxes - {'paint_degradation'}
  concrete-temperature-crack.jpg: 4 boxes - {'spalling'}
  images.jpg: 5 boxes - {'potholes'}
  images1.jpg: 1 boxes - {'corrosion'}
  images12.jpg: 4 boxes - {'potholes'}
  images123.jpg: 3 boxes - {'potholes', 'corrosion', 'spalling'}
  images21.jpg: 1 boxes - {'cracks'}
  images234.jpg: 10 boxes - {'cracks'}
  images5.jpg: 0 boxes - no detections
  images99.jpg: 3 boxes - {'spalling'}
  MicrosoftTeams-image_32.jpg: 1 boxes - {'potholes'}
  Newport_Whitepit_Lane_pot_hole.JPG: 3 boxes - {'potholes'}
  shutterstock_1667846680-scaled.jpg: 3 boxes - {'cracks'}
  What-is-Spalling-concrete.jpeg.png: 3 boxes - {'paint_degradation'}
Saved unseen_predictions_efficientdet_d0.png


<Figure size 1500x2000 with 15 Axes>

Faster R-CNN:
  close-up-of-a-weathered-wall-with-peeling-paint-revealing-layers-of-faded-colors-and-textured-surfaces-photo.jpg: 19 boxes - {'potholes', 'spalling', 'cracks', 'paint_degradation'}
  Concrete-Spalling.jpg: 12 boxes - {'spalling'}
  concrete-temperature-crack.jpg: 7 boxes - {'potholes', 'cracks'}
  images.jpg: 9 boxes - {'potholes', 'corrosion', 'spalling'}
  images1.jpg: 0 boxes - no detections
  images12.jpg: 6 boxes - {'potholes', 'cracks'}
  images123.jpg: 6 boxes - {'paint_degradation', 'potholes', 'cracks', 'spalling'}
  images21.jpg: 2 boxes - {'corrosion'}
  images234.jpg: 26 boxes - {'corrosion'}
  images5.jpg: 14 boxes - {'cracks'}
  images99.jpg: 9 boxes - {'cracks'}
  MicrosoftTeams-image_32.jpg: 7 boxes - {'potholes'}
  Newport_Whitepit_Lane_pot_hole.JPG: 6 boxes - {'potholes'}
  shutterstock_1667846680-scaled.jpg: 8 boxes - {'corrosion'}
  What-is-Spalling-concrete.jpeg.png: 6 boxes - {'cracks', 'spalling'}
Saved unseen_predictions_faster_r_cnn.png
Unseen i

In [33]:
# Run quantitative unseen checks only when labels are available
unseen_lbls = list(UNSEEN_LBL_DIR.glob('*.txt'))

if unseen_imgs and unseen_lbls:
    print(f'Ground-truth labels found ({len(unseen_lbls)}), computing metrics for all models\n')

    unseen_summary_rows = []
    for model_name, predict_fn in ALL_MODELS:
        u_iou = collect_iou_scores(predict_fn, UNSEEN_IMG_DIR, UNSEEN_LBL_DIR)
        _, _, u_fp, u_fn = collect_fp_fn(predict_fn, UNSEEN_IMG_DIR, UNSEEN_LBL_DIR)
        all_ious = [v for vals in u_iou.values() for v in vals]
        row = {
            'Model': model_name,
            'Total FP': sum(u_fp.values()),
            'Total FN': sum(u_fn.values()),
            'Mean IoU (matched)': round(np.mean(all_ious), 3) if all_ious else None,
            'Matched preds': len(all_ious),
        }
        unseen_summary_rows.append(row)
        print(f"{model_name}: FP={row['Total FP']}  FN={row['Total FN']}  "
              f"mean IoU={row['Mean IoU (matched)']}")

    df_unseen = pd.DataFrame(unseen_summary_rows).set_index('Model')
    print('\nUnseen data summary:')
    print(df_unseen.to_string())
    df_unseen.to_csv(RUNS_DIR / 'unseen_summary.csv')
    print('Saved unseen_summary.csv')

elif unseen_imgs:
    print('No labels for unseen images, qualitative inference only.')
else:
    print('No unseen images found.')

No labels for unseen images, qualitative inference only.


## 11. XAI - Backbone Activation Heatmaps

Explainability analysis: forward hooks extract feature map activations from each model's backbone
and visualise them as heatmaps overlaid on the input image.
Brighter regions indicate stronger model attention for that spatial location.
This shows *where* each model looks when detecting defects, not just *whether* it detects them.

In [34]:
def _get_heatmap(target_layer, run_forward, img_path, out_hw=None):
    """
    Hook target_layer during a forward pass and return a normalised activation heatmap.
    out_hw: (height, width) to resize the heatmap to; defaults to original image size.
    """
    _buf = []

    def _hook(m, inp, out):
        t = out[0] if isinstance(out, (list, tuple)) else out
        _buf.append(t.detach().cpu().float())

    handle = target_layer.register_forward_hook(_hook)
    img = Image.open(str(img_path)).convert('RGB')
    orig_w, orig_h = img.size
    t_in = TF.to_tensor(img.resize((IMG_SIZE, IMG_SIZE))).unsqueeze(0).to(DEVICE)
    try:
        with torch.no_grad():
            run_forward(t_in)
    except Exception:
        pass
    handle.remove()

    if not _buf:
        return None
    act = _buf[0].squeeze(0) if _buf[0].dim() == 4 else _buf[0]
    cam = act.abs().mean(0).numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    h, w = out_hw if out_hw else (orig_h, orig_w)
    return cv2.resize(cam, (w, h))


def _blend_heatmap(img_np, cam, alpha=0.45):
    """Overlay a jet-coloured heatmap on an RGB image array."""
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return ((1 - alpha) * img_np + alpha * heat).clip(0, 255).astype(np.uint8)


# Build XAI configs: model_name -> (forward_fn, target_layer)
_xai = {}

for _n, _m in YOLO_MODELS:
    try:
        from ultralytics.nn.modules import Detect
        _ls = list(_m.model.model)
        _last_feat = next(i for i in range(len(_ls) - 1, -1, -1)
                          if not isinstance(_ls[i], Detect))
        _tgt = _ls[max(0, _last_feat - 2)]
        _nn = _m.model
        _xai[_n] = (lambda t, nn=_nn: nn(t), _tgt)
        print(f'XAI ready: {_n}')
    except Exception as e:
        print(f'XAI skipped for {_n}: {e}')

if frcnn_model is not None:
    try:
        _tgt = frcnn_model.backbone.body.layer4[-1]
        _xai['Faster R-CNN'] = (lambda t: frcnn_model([t.squeeze(0)]), _tgt)
        print('XAI ready: Faster R-CNN')
    except Exception as e:
        print(f'XAI skipped for Faster R-CNN: {e}')

if effdet_model is not None:
    try:
        _tgt = list(effdet_model.model.backbone.blocks[-1])[-1]
        _xai['EfficientDet-D0'] = (lambda t: effdet_model(t), _tgt)
        print('XAI ready: EfficientDet-D0')
    except Exception as e:
        print(f'XAI skipped for EfficientDet-D0: {e}')

print(f'\nTotal XAI configs: {list(_xai.keys())}')

# Pick 4 annotated test images (label file must exist and be non-empty)
_lbl_test = DATA_DIR / 'labels' / 'test'
_xai_imgs = [
    p for p in sorted((DATA_DIR / 'images' / 'test').glob('*.jpg'))
    if (_lbl_test / (p.stem + '.txt')).exists()
    and (_lbl_test / (p.stem + '.txt')).stat().st_size > 0
][:4]

print(f'Using {len(_xai_imgs)} images for XAI visualisation')

n_m = len(_xai)
n_i = len(_xai_imgs)
fig, axes = plt.subplots(n_i, n_m + 1, figsize=(4 * (n_m + 1), 4 * n_i))
if n_i == 1:
    axes = [axes]

for row, img_path in enumerate(_xai_imgs):
    img_np = np.array(Image.open(str(img_path)).convert('RGB'))
    h_px, w_px = img_np.shape[:2]

    # Column 0: original image with ground-truth boxes
    axes[row][0].imshow(img_np)
    lbl_path = _lbl_test / (img_path.stem + '.txt')
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
        x1 = (cx - bw / 2) * w_px
        y1 = (cy - bh / 2) * h_px
        rect = patches.Rectangle((x1, y1), bw * w_px, bh * h_px,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        axes[row][0].add_patch(rect)
        axes[row][0].text(x1, y1 - 4, CLASSES[cls],
                          color='lime', fontsize=7, fontweight='bold')
    axes[row][0].axis('off')
    if row == 0:
        axes[row][0].set_title('Ground Truth', fontsize=9, fontweight='bold')

    # One column per model: activation heatmap overlay
    for col, (mname, (fwd_fn, tgt_layer)) in enumerate(_xai.items(), start=1):
        cam = _get_heatmap(tgt_layer, fwd_fn, img_path, out_hw=(h_px, w_px))
        if cam is not None:
            axes[row][col].imshow(_blend_heatmap(img_np, cam))
        else:
            axes[row][col].imshow(img_np)
            axes[row][col].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                 transform=axes[row][col].transAxes, fontsize=10)
        axes[row][col].axis('off')
        if row == 0:
            axes[row][col].set_title(mname, fontsize=9, fontweight='bold')

plt.suptitle('XAI: Backbone Activation Heatmaps - All Models (Brighter = Higher Attention)',
             fontsize=12)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'xai_activation_heatmaps.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved xai_activation_heatmaps.png')

# Per-model summary: average heatmap across all 4 images
fig2, axes2 = plt.subplots(1, n_m, figsize=(4 * n_m, 4))
if n_m == 1:
    axes2 = [axes2]

for ax, (mname, (fwd_fn, tgt_layer)) in zip(axes2, _xai.items()):
    avg_cam = None
    count = 0
    for img_path in _xai_imgs:
        img_np = np.array(Image.open(str(img_path)).convert('RGB'))
        h_px, w_px = img_np.shape[:2]
        cam = _get_heatmap(tgt_layer, fwd_fn, img_path, out_hw=(h_px, w_px))
        if cam is not None:
            avg_cam = cam if avg_cam is None else avg_cam + cam
            count += 1
    if avg_cam is not None and count > 0:
        avg_cam /= count
        ax.imshow(avg_cam, cmap='jet')
        ax.set_title(f'{mname}\navg attention', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.suptitle('Average Activation Intensity per Model (4 Test Images)', fontsize=11)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'xai_avg_attention.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved xai_avg_attention.png')

XAI ready: YOLOv11s
XAI ready: YOLOv8s
XAI ready: RT-DETR-L
XAI ready: YOLOv8n
XAI ready: YOLOv11n
XAI ready: Faster R-CNN
XAI ready: EfficientDet-D0

Total XAI configs: ['YOLOv11s', 'YOLOv8s', 'RT-DETR-L', 'YOLOv8n', 'YOLOv11n', 'Faster R-CNN', 'EfficientDet-D0']
Using 4 images for XAI visualisation


<Figure size 3200x1600 with 32 Axes>

Saved xai_activation_heatmaps.png


<Figure size 2800x400 with 7 Axes>

Saved xai_avg_attention.png
